# Data Cleaning and Filtering
This notebook focuses on cleaning and filtering the house sales dataset. Common steps include handling missing values, removing duplicates, correcting data types, and filtering for relevant records.

In [1]:
import pandas as pd
# Load the data
df = pd.read_csv('data/eda.csv')
df.head()

,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,...,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,price,date
0,7129300520,3.0,1.00,1180.0,5650.0,1.0,NaN,0.0,3,7,...,0.0,1955,0.0,98178,47.5112,-122.257,1340.0,5650.0,221900.0,2014-10-13
1,6414100192,3.0,2.25,2570.0,7242.0,2.0,0.0,0.0,3,7,...,400.0,1951,19910.0,98125,47.7210,-122.319,1690.0,7639.0,538000.0,2014-12-09
2,5631500400,2.0,1.00,770.0,10000.0,1.0,0.0,0.0,3,6,...,0.0,1933,NaN,98028,47.7379,-122.233,2720.0,8062.0,180000.0,2015-02-25
3,2487200875,4.0,3.00,1960.0,5000.0,1.0,0.0,0.0,5,7,...,910.0,1965,0.0,98136,47.5208,-122.393,1360.0,5000.0,604000.0,2014-12-09
4,1954400510,3.0,2.00,1680.0,8080.0,1.0,0.0,0.0,3,8,...,0.0,1987,0.0,98074,47.6168,-122.045,1800.0,7503.0,510000.0,2015-02-18


## Initial Data Overview
Check for missing values, duplicates, and data types.

In [2]:
print(df.info())
print(df.isnull().sum())
print(f'Duplicates: {df.duplicated().sum()}')
print(df.dtypes)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21597 entries, 0 to 21596
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             21597 non-null  int64  
 1   bedrooms       21597 non-null  float64
 2   bathrooms      21597 non-null  float64
 3   sqft_living    21597 non-null  float64
 4   sqft_lot       21597 non-null  float64
 5   floors         21597 non-null  float64
 6   waterfront     19206 non-null  float64
 7   view           21534 non-null  float64
 8   condition      21597 non-null  int64  
 9   grade          21597 non-null  int64  
 10  sqft_above     21597 non-null  float64
 11  sqft_basement  21145 non-null  float64
 12  yr_built       21597 non-null  int64  
 13  yr_renovated   17749 non-null  float64
 14  zipcode        21597 non-null  int64  
 15  lat            21597 non-null  float64
 16  long           21597 non-null  float64
 17  sqft_living15  21597 non-null  float64
 18  sqft_l

## Remove Duplicates

In [3]:
df = df.drop_duplicates().copy()
print(f'After removing duplicates: {df.shape}')

After removing duplicates: (21597, 21)


## Handle Missing Values
Drop rows with missing values in key columns (e.g., price, zipcode, date).

In [ ]:
df = df.dropna(subset=['price', 'zipcode', 'date']).copy()
print(f'After dropping missing key values: {df.shape}')

## Convert Data Types
Ensure correct types for price, date, and zipcode.

In [4]:
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['zipcode'] = df['zipcode'].astype(str).str.strip().replace({'nan': None})
df = df.dropna(subset=['price', 'date', 'zipcode']).copy()
print(df.dtypes)

id                        int64
bedrooms                float64
bathrooms               float64
sqft_living             float64
sqft_lot                float64
floors                  float64
waterfront              float64
view                    float64
condition                 int64
grade                     int64
sqft_above              float64
sqft_basement           float64
yr_built                  int64
yr_renovated            float64
zipcode                  object
lat                     float64
long                    float64
sqft_living15           float64
sqft_lot15              float64
price                   float64
date             datetime64[ns]
dtype: object


## Filter for Central Seattle Zipcodes
Focus on central Seattle zipcodes as an example.

In [5]:
central_zips = ["98101", "98102", "98103", "98104", "98105", "98109", "98112", "98121", "98122", "98144"]
df_central = df[df['zipcode'].isin(central_zips)].copy()
print(df_central['zipcode'].value_counts())

zipcode
98103    602
98144    343
98122    290
98112    269
98105    229
98109    109
98102    104
Name: count, dtype: int64


## Filter by Geolocation

Remove properties that are geographically outside of Seattle's approximate bounding box. This helps to remove outliers or incorrectly coded locations.

In [7]:

# Define an approximate bounding box for Seattle
seattle_lat_bounds = (47.4, 47.8)
seattle_lon_bounds = (-122.5, -122.1)

# Identify lat/lon columns
lat_col = 'lat' if 'lat' in df.columns else 'latitude'
lon_col = 'long' if 'long' in df.columns else 'longitude'

if lat_col in df.columns and lon_col in df.columns:
    # Ensure lat/lon are numeric
    df[lat_col] = pd.to_numeric(df[lat_col], errors='coerce')
    df[lon_col] = pd.to_numeric(df[lon_col], errors='coerce')
    
    # Filter based on bounding box
    original_rows = len(df)
    df = df[
        (df[lat_col] >= seattle_lat_bounds[0]) & (df[lat_col] <= seattle_lat_bounds[1]) &
        (df[lon_col] >= seattle_lon_bounds[0]) & (df[lon_col] <= seattle_lon_bounds[1])
    ].copy()
    
    print(f"Removed {original_rows - len(df)} rows outside of Seattle's bounding box.")
    print(f"Remaining rows: {len(df)}")
else:
    print("Latitude/longitude columns not found; skipping geolocation filtering.")

# Save the final cleaned data
df.to_csv('data/eda_cleaned_geo.csv', index=False)


Removed 7110 rows outside of Seattle's bounding box.
Remaining rows: 14487


## Save Cleaned Data

In [6]:
df.to_csv('data/eda_cleaned.csv', index=False)
df_central.to_csv('data/eda_central.csv', index=False)